# post-trade

Sanity check on a `micro-recorder` capture: every feature column is landing and rows key on
`event_ts_us`. Run any capture, Ctrl-C it (files seal on close), then run all cells.

Setup: `uv sync` in `strategies/`, pick the `.venv` kernel. Data plumbing (sealed-file
discovery, sim/live layout, footer dictionaries, the wide pivot) lives in `etl.py`. An
in-flight run's open file has no footer yet — it is skipped automatically. For the markout /
toxic-flow deep dive see `strat-micro-recorder/markouts.ipynb`.

In [ ]:
import polars as pl

import etl

RUN = etl.find_run("../data", "strat-micro-recorder", "te-binance-spot-btcusdt")
features, footer, empty = etl.load_features(RUN)

print("run      ", RUN)
print("strategy ", footer["strategy_id"], " mode", footer["execution_mode"])
print("scale    ", footer["fixed_scale"], " engine", footer["engine_version"])
print("features ", len(footer["feature_dictionary"]), "declared,", len(empty), "never emitted:", empty)
print("instruments", footer["instrument_dictionary"])
features

In [ ]:
# Every declared feature should appear here with a sane value range. The volatilities lag the
# start of a capture — the EGARCH fit needs closed 1m candles behind it.
features.group_by("feature").agg(
    pl.len().alias("rows"),
    pl.col("value").min().alias("min"),
    pl.col("value").max().alias("max"),
).sort("feature")

In [ ]:
# One row per tick, one column per instrument+feature — the shape a model would read. Columns
# come from the combos the capture actually produced, in footer-dictionary order. A combo the
# capture never produced never appears at all.
wide = etl.pivot_wide(features, footer)
wide

In [ ]:
# CSV for Excel. Space-separated datetimes to 3dp — Excel parses that; the ISO "T" form it
# imports as text. Nulls become empty cells. Lands beside the run it came from, under
# ../data/, which .gitignore keeps out of the repo.
export = wide
out = RUN / "wide.csv"
export.write_csv(out, datetime_format="%Y-%m-%d %H:%M:%S%.3f")

print(f"{out.resolve()}\n{export.height:,} rows x {export.width} cols, {out.stat().st_size / 1e6:.1f} MB")
if export.height > 1_048_576:
    print("WARNING: over Excel's row limit — it will silently truncate. Filter or downsample first.")

In [ ]:
# The polymarket engine records rotations only (market handoffs), and it records them live.
poly = etl.find_run("../data", "strat-micro-recorder", "te-polymarket-btc-updown-5m")
rotations, _ = etl.load_table(poly, "rotations")
rotations.sort("received_ts_us")